# Regressão Simbólica — Redescoberta de Leis Físicas

## O que é regressão simbólica

Na regressão clássica você define a forma da equação antes de ajustá-la — por exemplo, assume que a relação é linear e encontra os coeficientes `a` e `b` de `y = ax + b`. A forma funcional é uma escolha sua, não do algoritmo.

A regressão simbólica não parte de nenhuma suposição sobre a forma. Dados os valores de entrada e saída, o algoritmo busca simultaneamente a estrutura matemática da equação e seus coeficientes. O resultado é uma expressão legível — como `y = 3.14·x²` — em vez de uma caixa preta.

## Como o PySR funciona

O PySR usa um algoritmo genético para evoluir uma população de expressões matemáticas. Cada expressão é representada como uma árvore de operações — nós internos são operadores (`+`, `×`, `sin`) e folhas são variáveis ou constantes. O algoritmo avalia cada expressão pelo erro nos dados, seleciona as melhores, e as combina ou modifica para gerar novas expressões. Após muitas gerações, as equações convergem para formas que descrevem bem os dados.

## O problema da complexidade — fronteira de Pareto

Uma equação com muitos termos pode ajustar qualquer conjunto de dados perfeitamente, mas não revela nenhuma lei física — é overfitting simbólico. O PySR equilibra dois objetivos simultaneamente: minimizar o erro de ajuste e minimizar a complexidade da expressão. As equações que não podem melhorar em um objetivo sem piorar no outro formam a **fronteira de Pareto**. O algoritmo retorna essa fronteira completa, e cabe ao pesquisador escolher o ponto que faz mais sentido físico.


## Problema 1 — Queda livre

### Objetivo
Redescobrir a equação da posição em função do tempo para um objeto em queda livre:

$$y(t) = y_0 - \frac{1}{2}g t^2$$

### Escolha dos operadores
Como a equação esperada é um polinômio de grau 2 em `t`, não há justificativa física para incluir funções como `sin`, `exp` ou `log`. Os operadores foram limitados a `+`, `-`, `*` e `square` — o mínimo necessário para representar a lei correta. Essa restrição também reduz o espaço de busca e o tempo de execução, o que é relevante dado o hardware disponível.

### Parâmetros do modelo
O número de iterações foi mantido baixo (40) pois a equação é simples e o algoritmo converge rapidamente. O `maxsize=15` é mais que suficiente para uma expressão com poucos termos. Os demais parâmetros foram mantidos no padrão do PySR.

### Limitação: a constante gravitacional
Os dados foram gerados inteiramente para a Terra, onde `g = 9.81 m/s²` é constante. Sem variação nesse valor, o modelo não consegue identificar `g` como grandeza física independente — ele absorve o valor numérico diretamente no coeficiente encontrado. Para isolar `g` seria necessário dados de múltiplos corpos celestes com gravidades diferentes, como Marte, Lua e Júpiter.

### Outras limitações
- Apenas uma altura inicial foi simulada, então `y₀` aparece como constante fixa
- Não foi definido um limite para a queda — o objeto atravessa o chão
- É uma simplificação do problema real, mas serve como primeira validação do método

### Resultado esperado
O modelo deve encontrar algo próximo de `y = 100 - 4.9 * t²`, onde 4.9 ≈ g/2.

In [27]:
import numpy as np
from pysr import PySRRegressor

data = np.loadtxt("../data/raw/dados_queda_livre.csv", delimiter=",", skiprows=1)
t = data[:, 0]
y = data[:, 1]




model = PySRRegressor(
    model_selection="best", # Foca na precisão em vez de apenas simplicidade
    niterations=100,
    
    #operadores
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["square"],
    
    temp_equation_file=True,
    delete_tempfiles=True,
    output_directory=None, # Tenta não criar a pasta 'outputs'
    
    # para o ruido, ainda falta melhorar
    # weights=1.0 / (np.full_like(y, 0.1)**2), 
    
    #complexidade e parsimônia
    maxsize=20, # Limita o tamanho da equação para não "alucinar" no ruído
    #parsimony=0.001,
    
    variable_names=["t", "G"]
)

model.fit(X, y)

print("\nEquação com relação de variáveis:")
print(model.get_best().equation)

c:\projeto\symbolic_regression_physics\.venv\Lib\site-packages\pysr\sr.py:1046: FutureWarning: `variable_names` is a data-dependent parameter and should be passed when fit is called. Ignoring parameter; please pass `variable_names` during the call to fit instead.
  warnings.warn(
c:\projeto\symbolic_regression_physics\.venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(



Expressions evaluated per second: 4.420e+05
Progress: 2220 / 3100 total iterations (71.613%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.157e+04  0.000e+00  y = -63.611
3           9.654e+03  4.009e-01  y = x₀ * -21.857
4           4.474e+03  7.681e-01  y = square(x₀) * -3.2397
5           1.371e+03  1.183e+00  y = (x₀ * -48.992) + 181.35
6           3.532e+00  5.961e+00  y = (square(x₀) * -4.8999) - -100.13
7           3.532e+00  1.192e-07  y = (x₀ * (x₀ * -4.9)) + 100.13
8           3.524e+00  2.177e-03  y = (square(x₀ + -0.012196) + -20.344) * -4.9112
9           3.524e+00  9.537e-07  y = (x₀ * ((x₀ * -4.9112) + 0.11997)) + 99.91
11          3.524e+00  1.788e-07  y = (((x₀ * -4.9112) * x₀) + 99.909) + (x₀ * 0.12045)
13          3.378e+00  2.124e-02  y = (0.084523 / (x₀ + 

## Problema 2 — Oscilador harmônico

### Objetivo
Redescobrir a equação da posição em função do tempo para uma massa presa a uma mola:

$$x(t) = A \cos(\omega t)$$

onde $A$ é a amplitude e $\omega = \sqrt{k/m}$ é a frequência angular natural do sistema.

### Escolha dos operadores
A solução do oscilador harmônico é inerentemente oscilatória — diferente da queda livre, aqui `cos` é fisicamente necessário. O operador `square` foi mantido pois pode aparecer em representações equivalentes. Divisão foi incluída pois $\omega$ pode emergir como razão entre parâmetros. Operadores como `exp`, `log` e `sin` foram excluídos por não terem justificativa física para esse sistema.

### Parâmetros do modelo
O número de iterações foi aumentado para 100 em relação à queda livre — a equação é mais complexa e o espaço de busca maior. O `maxsize=20` permite expressões com mais termos, necessário para capturar o produto $A \cdot \cos(\omega t)$.

### Limitação: amplitude e frequência como constantes
Assim como `g` na queda livre, $A$ e $\omega$ não aparecem como variáveis independentes — os dados foram gerados com valores fixos de `k`, `m` e condição inicial. O modelo vai absorver esses valores como coeficientes numéricos. Para descobrir $\omega = \sqrt{k/m}$ como relação física seria necessário variar `k` e `m` entre experimentos e passar essas grandezas como colunas do dataset.

### Resultado esperado
O modelo deve encontrar algo próximo de `x = 0.1 * cos(3.16 * t)`, onde $0.1$ é a amplitude e $3.16 \approx \sqrt{10/1} = \omega$.

In [28]:
import numpy as np
from pysr import PySRRegressor

data = np.loadtxt("../data/raw/dados_massa_mola.csv", delimiter=",", skiprows=1)
t = data[:, 0]
y = data[:, 1]

#reshape
t = t.reshape(-1, 1)


model = PySRRegressor(
    model_selection="best", # Foca na precisão em vez de apenas simplicidade
    niterations=100,
    
    #operadores
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["square", "cos"],
    
    temp_equation_file=True,
    delete_tempfiles=True,
    output_directory=None, # Tenta não criar a pasta 'outputs'
    
    # para o ruido, ainda falta melhorar
    # weights=1.0 / (np.full_like(y, 0.1)**2), 
    
    #complexidade e parsimônia
    maxsize=20, # Limita o tamanho da equação para não "alucinar" no ruído
    #parsimony=0.001
   
)

model.fit(t, y)
print("\nEquação com relação de variáveis:")
print(model.get_best().equation)

c:\projeto\symbolic_regression_physics\.venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(



Expressions evaluated per second: 3.320e+05
Progress: 2108 / 3100 total iterations (68.000%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.115e-03  0.000e+00  y = 0.0009906
3           5.114e-03  7.749e-05  y = x₀ * 0.00023157
4           5.111e-03  5.779e-04  y = square(x₀) * 4.8895e-05
5           4.961e-03  2.977e-02  y = 0.0058166 / (x₀ + 0.052382)
6           2.543e-05  5.273e+00  y = cos(x₀ * -3.1609) * 0.10042
8           2.513e-05  5.943e-03  y = cos((0.99814 - x₀) / 0.31614) * -0.10035
14          2.327e-05  1.281e-02  y = cos(((x₀ + (0.086801 / (x₀ + 1.2615))) - 1.0316) / 0.3...
                                      1497) * -0.10052
20          2.299e-05  2.031e-03  y = cos((((0.088532 / ((x₀ + 1.102) - (0.088427 / (x₀ + -2...
                                      .0